# Building a Neural netwok using single perceptron

In [18]:
import torch 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Library used for data preprocessing.

In [19]:
from sklearn.preprocessing import StandardScaler , OneHotEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split,cross_val_score,cross_validate , KFold ,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report ,confusion_matrix

### This dataset is used for train and test our neural network.

In [20]:
df = pd.read_csv(r"C:\Users\ASUS\OneDrive\Desktop\Program\AI ML\heart.csv")
df.sample(5)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
692,39,F,NAP,94,199,0,Normal,179,N,0.0,Up,0
30,53,M,NAP,145,518,0,Normal,130,N,0.0,Flat,1
651,61,M,ASY,140,207,0,LVH,138,Y,1.9,Up,1
732,56,F,ASY,200,288,1,LVH,133,Y,4.0,Down,1
821,60,F,NAP,102,318,0,Normal,160,N,0.0,Up,0


In [21]:
categorical_column = ['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope']
numric_columns = df.drop(columns=['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope','HeartDisease'] ,axis=1).columns.tolist()

### Numric data Pipeline

In [22]:
Numric_pipeline = Pipeline([
    ("SimpleImputer",SimpleImputer(strategy='median')),
    ("Scaler",StandardScaler())
])

### Categorical Data Pipeline.

In [23]:
categorical_pipeline =Pipeline([
    ("ohe",OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

### Full pipeline

In [24]:
complete_pipeline = ColumnTransformer([
    ("num",Numric_pipeline,numric_columns),
    ("cat",categorical_pipeline,categorical_column)
])

In [25]:
target = df['HeartDisease']
final_df = df.drop(['HeartDisease'],axis=1)

In [26]:
process_data = complete_pipeline.fit_transform(final_df)
process_data

array([[-1.4331398 ,  0.41090889,  0.82507026, ...,  0.        ,
         0.        ,  1.        ],
       [-0.47848359,  1.49175234, -0.17196105, ...,  0.        ,
         1.        ,  0.        ],
       [-1.75135854, -0.12951283,  0.7701878 , ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [ 0.37009972, -0.12951283, -0.62016778, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.37009972, -0.12951283,  0.34027522, ...,  0.        ,
         1.        ,  0.        ],
       [-1.64528563,  0.30282455, -0.21769643, ...,  0.        ,
         0.        ,  1.        ]])

In [27]:
process_data.shape , target.shape

((918, 20), (918,))

### Spliting data set for training and testing 
### Training = 70%
### Testing = 30%

In [28]:
x_train,x_test,y_train,y_test = train_test_split(process_data, target, random_state=42, test_size=0.3, stratify= target)

## Convert numpy data  into tensor data 

In [29]:
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test, dtype= torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy())
y_test_tesnor = torch.tensor(y_test.to_numpy())

# Let's build our model.

In [42]:
class perceptron:
    def __init__(self,lt = 1e-1):
        self.w = None
        self.b = None
        self.lt = lt

    #forward pass.
    def forward(self,X):
        z = torch.matmul(X,self.w) + self.b
        return torch.sigmoid(z)
    

    
    # Define Log loss Function.
    def log_loss(self, y_true, y_predicted):
        epsilon = 1e-10

        # Clamp values to avoid log(0)
        y_predicted = torch.clamp(y_predicted, epsilon, 1 - epsilon)
    
        loss = -torch.mean(y_true * torch.log(y_predicted) + (1 - y_true) * torch.log(1 - y_predicted))
        return loss



    # Define Fit Function.
    def fit(self, X ,Y, epochs):

        prev_loss = float('inf')
        accuracy = 0

        self.w = torch.rand(X.shape[1], dtype= torch.float32 ,requires_grad= True)
        self.b = torch.zeros(1, dtype = torch.float32, requires_grad=True)

        for i in range(epochs):
            # forward pass.
            y_pred = self.forward(X)

            # calculating loss.
            curr_loss = self.log_loss(Y,y_pred)

            # Claculating Accuracy.
            y_pred = torch.round(y_pred)
            accuracy = ((y_pred == Y).sum().item() * 100) / len(Y)

            curr_loss.backward()

            with torch.no_grad():
                self.w -= self.lt * self.w.grad
                self.b -= self.lt * self.b.grad

            self.w.grad.zero_()
            self.b.grad.zero_()

            if (prev_loss - curr_loss) < 1e-5:
                break

            prev_loss = curr_loss

            if i % 10 == 0:
                print(f"epoch : {i} || loss : {curr_loss:.4f} || accuracy : {accuracy:.4f}")
        return prev_loss.item(),accuracy


    
    #Define predict function.
    def predict(self,X):
        z = torch.matmul(X, self.w) + self.b
        return torch.sigmoid(z)
        # return self.forward(X)

        
    #Define evaluate function.
    def evaluate(self, X, Y):
        y_pred = self.predict(X)
        loss = self.log_loss(Y, y_pred)
        
        y_pred = torch.round(y_pred)
        accuracy = ((y_pred == Y).sum().item() * 100) / len(Y)
        return loss.item() ,accuracy


In [43]:
model = perceptron()


### Model training 

In [44]:
model.fit(x_train_tensor,y_train_tensor,1000)

epoch : 0 || loss : 1.2709 || accuracy : 50.6231
epoch : 10 || loss : 0.8896 || accuracy : 56.5421
epoch : 20 || loss : 0.6937 || accuracy : 64.6417
epoch : 30 || loss : 0.5875 || accuracy : 72.1184
epoch : 40 || loss : 0.5234 || accuracy : 74.4548
epoch : 50 || loss : 0.4814 || accuracy : 77.7259
epoch : 60 || loss : 0.4523 || accuracy : 79.5950
epoch : 70 || loss : 0.4314 || accuracy : 80.6854
epoch : 80 || loss : 0.4158 || accuracy : 81.9315
epoch : 90 || loss : 0.4038 || accuracy : 83.0218
epoch : 100 || loss : 0.3945 || accuracy : 83.4891
epoch : 110 || loss : 0.3869 || accuracy : 83.8006
epoch : 120 || loss : 0.3808 || accuracy : 83.9564
epoch : 130 || loss : 0.3757 || accuracy : 84.2679
epoch : 140 || loss : 0.3715 || accuracy : 84.2679
epoch : 150 || loss : 0.3678 || accuracy : 84.4237
epoch : 160 || loss : 0.3647 || accuracy : 84.2679
epoch : 170 || loss : 0.3620 || accuracy : 84.2679
epoch : 180 || loss : 0.3596 || accuracy : 84.2679
epoch : 190 || loss : 0.3575 || accuracy :

(0.3357146084308624, 85.51401869158879)

### After training, we obtained a loss of 0.3332 and an accuracy of 84.89%.

In [50]:
model.evaluate(x_test_tensor,y_test_tesnor)

(0.31762760877609253, 88.04347826086956)

### During evaluation, the model achieved a loss of 0.3224 and an accuracy of 88.4%.

# Classification report 

In [51]:
y_predicted = model.predict(x_test_tensor)
y_predicted = torch.round(y_predicted)

In [52]:
print(classification_report(y_test_tesnor.detach().numpy(),y_predicted.detach().numpy()))

              precision    recall  f1-score   support

           0       0.89      0.84      0.86       123
           1       0.88      0.92      0.89       153

    accuracy                           0.88       276
   macro avg       0.88      0.88      0.88       276
weighted avg       0.88      0.88      0.88       276

